# BdSLW60 - Week 1 pipeline


In [1]:
# mediapipe >= 0.10.31 removed the legacy mp.solutions.holistic API (and 0.10.21 has no wheel for Python 3.13),
# so this notebook uses the Tasks HolisticLandmarker, which needs a model file.
!pip install -q mediapipe
import os, sys, urllib.request
print("python", sys.version.split()[0])
MODEL_PATH = '/kaggle/working/holistic_landmarker.task'
MODEL_URL = ('https://storage.googleapis.com/mediapipe-models/holistic_landmarker/'
             'holistic_landmarker/float16/latest/holistic_landmarker.task')
if not os.path.exists(MODEL_PATH):
    try:
        urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
    except Exception as e:
        print('MODEL DOWNLOAD FAILED:', e, '\nDownload holistic_landmarker.task by hand, add it as a Kaggle dataset '
              'and set MODEL_PATH to its /kaggle/input/... path.')
if os.path.exists(MODEL_PATH):
    print('model file: %s (%.1f MB)' % (MODEL_PATH, os.path.getsize(MODEL_PATH) / 1e6))


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
python 3.13.15
model file: /kaggle/working/holistic_landmarker.task (13.7 MB)


In [2]:
import os, re, json, time, math, random, collections, subprocess, sys, shutil, hashlib
from pathlib import Path
import numpy as np
import pandas as pd

WORK = Path('/kaggle/working')
CACHE = WORK / 'bdslw60_cache'
KP_DIR = CACHE / 'keypoints'
for d in (CACHE, KP_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Saved keypoints from an earlier run (a Kaggle Dataset made from bdslw60_cache), e.g.
# Path('/kaggle/input/bdslw60-cache/bdslw60_cache/keypoints'). None = extract from the videos.
KP_SOURCE = '/kaggle/input/notebooks/rishta19/bdslw60-week-1/bdslw60_cache/keypoints'

T_OUT = 64              # frames after resampling
MAX_GAP = 5             # longest gap (frames) that is interpolated
N_NODES = 49            # 7 body + 21 left hand + 21 right hand
QUICK_TEST_VIDEOS = None  # e.g. 12 -> extract only 12 videos first to test everything quickly

TEST_SIGNERS = (4, 8)            # official test signers: never evaluated before week 3
P1_VAL_SIGNER = 5                # checkpoint selection for the final P1 runs
INNER_VAL_SIGNERS = (1, 10, 13)  # model selection: the 3 signers with the most left-handed clips
MIN_VAL_CLIPS = 500              # smallest signer allowed as a LOSO validation signer

# Word ids in class order (Table 4 of the paper): class index 0..59
WORD_IDS = (list(range(1, 13)) + [19, 20] + list(range(37, 51)) + list(range(91, 101))
            + [111, 112] + list(range(211, 221)) + list(range(351, 361)))
assert len(WORD_IDS) == 60
WORD2CLS = {w: i for i, w in enumerate(WORD_IDS)}

# Trials per signer from Table 6 of the paper (train + test columns), used as a sanity check
TABLE6 = {1: 975, 2: 744, 3: 583, 4: 637, 5: 600, 6: 815, 7: 107, 8: 639, 9: 656,
          10: 383, 11: 771, 12: 655, 13: 621, 14: 112, 15: 655, 16: 97, 17: 116, 18: 141}
assert sum(TABLE6.values()) == 9307
assert not set(INNER_VAL_SIGNERS) & set(TEST_SIGNERS) and P1_VAL_SIGNER not in TEST_SIGNERS

VID_RE = re.compile(r'U(\d+)W(\d+)([A-Za-z]*)', re.I)   # e.g. U11W1F.mp4
print('config ok')


config ok


In [3]:
# ---- Cell 3: find everything the dataset contains ----
INPUT_ROOT = Path('/kaggle/input')
assert INPUT_ROOT.exists() and any(INPUT_ROOT.iterdir()), 'No input found: use Add Input -> hasaniut/bdslw60'
all_files = [p for p in INPUT_ROOT.rglob('*') if p.is_file()]
print('files found:', len(all_files))
print('top-level inputs:', [p.name for p in INPUT_ROOT.iterdir()])
ext_count = collections.Counter(p.suffix.lower() for p in all_files)
print('extensions:', dict(ext_count.most_common(12)))
for ext in list(ext_count)[:8]:
    examples = [str(p) for p in all_files if p.suffix.lower() == ext][:3]
    print('\n' + (ext or '(no extension)') + ':')
    for e in examples:
        print('   ', e)

files found: 1626
top-level inputs: ['notebooks', 'datasets']
extensions: {'.mp4': 777, '.npz': 770, '.json': 33, '.txt': 30, '.npy': 5, '.pt': 2, '.csv': 2, '.pdf': 1, '.html': 1, '.task': 1, '.ipynb': 1, '.css': 1}

.pdf:
    /kaggle/input/datasets/hasaniut/bdslw60/BdSLW60.pdf

.mp4:
    /kaggle/input/datasets/hasaniut/bdslw60/W97-98/U12W98F.mp4
    /kaggle/input/datasets/hasaniut/bdslw60/W97-98/U11W98F.mp4
    /kaggle/input/datasets/hasaniut/bdslw60/W97-98/U13W98F.mp4

.json:
    /kaggle/input/datasets/hasaniut/bdslw60/W97-98/output1.json
    /kaggle/input/datasets/hasaniut/bdslw60/W93-94/output1.json
    /kaggle/input/datasets/hasaniut/bdslw60/W215-216/output1.json

.txt:
    /kaggle/input/datasets/hasaniut/bdslw60/W97-98/annotationW97-98.txt
    /kaggle/input/datasets/hasaniut/bdslw60/W93-94/annotationW93-94.txt
    /kaggle/input/datasets/hasaniut/bdslw60/W215-216/annotationW215-216.txt

.html:
    /kaggle/input/notebooks/rishta19/bdslw60-week-1/__results__.html

.task:
    /kaggl

In [4]:
# ---- Cell 4: videos (user, word, view come from the file name, e.g. U11W1F.mp4) ----
rows = []
for p in all_files:
    if p.suffix.lower() in ('.mp4', '.avi', '.mov', '.mkv'):
        m = VID_RE.search(p.stem)
        if m:
            rows.append(dict(path=str(p), stem=p.stem, user=int(m[1]), word=int(m[2]), view=m[3].upper()))
videos = pd.DataFrame(rows)
assert len(videos) > 0, 'No videos matched the U<user>W<word><view> naming - send me the output of cell 3.'
print('video files by view:', videos.view.value_counts().to_dict())
dropped = []                                     # (file, reason) for every video we do not use
if (videos.view == 'F').any():
    for r in videos[videos.view != 'F'].itertuples():
        dropped.append((r.path, 'view %s (not front)' % r.view))
    videos = videos[videos.view == 'F']
dup_mask = videos.duplicated(['user', 'word'], keep='first')
if dup_mask.any():
    print('WARNING:', int(dup_mask.sum()), 'duplicate (user, word) videos - keeping the first of each')
    for r in videos[dup_mask].itertuples():
        kept = videos[(videos.user == r.user) & (videos.word == r.word) & ~dup_mask].path.iloc[0]
        dropped.append((r.path, 'duplicate of %s' % kept))
    videos = videos[~dup_mask]
videos = videos.reset_index(drop=True)
print('front-view videos:', len(videos), '| signers:', videos.user.nunique(), '| words:', videos.word.nunique())
unknown_words = sorted(set(videos.word) - set(WORD_IDS))
print('words not in the 60-word list:', unknown_words[:20], '(these will be ignored)' if unknown_words else '')
for r in videos[~videos.word.isin(WORD_IDS)].itertuples():
    dropped.append((r.path, 'word not in the 60-word list'))
videos = videos[videos.word.isin(WORD_IDS)].reset_index(drop=True)
print('videos kept:', len(videos), '(60 words x 18 signers = 1080 would be complete)')
print('\nvideo files not used:')
for path, why in dropped:
    print('   %s  <- %s' % (path, why))


video files by view: {'F': 776, 'FL': 1}
front-view videos: 774 | signers: 18 | words: 60
words not in the 60-word list: [] 
videos kept: 774 (60 words x 18 signers = 1080 would be complete)

video files not used:
   /kaggle/input/datasets/hasaniut/bdslw60/W357-358/U18W358FL.mp4  <- view FL (not front)
   /kaggle/input/datasets/hasaniut/bdslw60/W357-358/U1W357F_1.mp4  <- duplicate of /kaggle/input/datasets/hasaniut/bdslw60/W357-358/U1W357F_2.mp4
   /kaggle/input/datasets/hasaniut/bdslw60/W357-358/U1W357F_3.mp4  <- duplicate of /kaggle/input/datasets/hasaniut/bdslw60/W357-358/U1W357F_2.mp4


In [5]:
# ---- Cell 5: look at the annotation files ----
json_files = sorted([p for p in all_files if p.suffix.lower() == '.json'])
txt_files = sorted([p for p in all_files if p.suffix.lower() == '.txt'])
print(len(json_files), 'json files;', len(txt_files), 'txt files')
for p in json_files[:5]:
    print('  ', p)

def describe(obj, depth=0, max_depth=5, max_items=4):
    pad = '  ' * depth
    if depth > max_depth:
        print(pad + '...')
        return
    if isinstance(obj, dict):
        print(pad + 'dict(%d keys)' % len(obj))
        for k in list(obj)[:max_items]:
            print(pad + '  key %r:' % (k,))
            describe(obj[k], depth + 2, max_depth, max_items)
        if len(obj) > max_items:
            print(pad + '  ... +%d more keys' % (len(obj) - max_items))
    elif isinstance(obj, list):
        print(pad + 'list(len=%d)' % len(obj))
        for v in obj[:2]:
            describe(v, depth + 1, max_depth, max_items)
    else:
        print(pad + repr(obj)[:80])

if json_files:
    with open(json_files[0], encoding='utf-8') as f:
        sample_json = json.load(f)
    print('\nSTRUCTURE OF', json_files[0].name)
    describe(sample_json)
if txt_files:
    print('\nFIRST LINES OF', txt_files[0].name)
    with open(txt_files[0], errors='ignore') as f:
        print(''.join(f.readlines()[:15]))

33 json files; 30 txt files
   /kaggle/input/datasets/hasaniut/bdslw60/W1-2/output1.json
   /kaggle/input/datasets/hasaniut/bdslw60/W11-12/output1.json
   /kaggle/input/datasets/hasaniut/bdslw60/W111-112/output1.json
   /kaggle/input/datasets/hasaniut/bdslw60/W19-20/output1.json
   /kaggle/input/datasets/hasaniut/bdslw60/W211-212/output1.json

STRUCTURE OF output1.json
dict(2 keys)
  key 'W1':
    dict(13 keys)
      key 'U1':
        dict(1 keys)
          key 'LeftHand':
            ...
      key 'U2':
        dict(1 keys)
          key 'RightHand':
            ...
      key 'U3':
        dict(1 keys)
          key 'RightHand':
            ...
      key 'U4':
        dict(1 keys)
          key 'LeftHand':
            ...
      ... +9 more keys
  key 'W2':
    dict(13 keys)
      key 'U1':
        dict(1 keys)
          key 'LeftHand':
            ...
      key 'U2':
        dict(1 keys)
          key 'RightHand':
            ...
      key 'U3':
        dict(1 keys)
          key 'Rig

In [6]:
# ---- Cell 6: turn the annotations into one table of trials (video, start frame, end frame, hand) ----
START_KEY = None   # set these three if the auto-guess printed below is wrong, e.g. 'actual_start'
END_KEY = None     # e.g. 'end'
HAND_KEY = None    # e.g. 'hand'
HAND_MAP = {}      # extra raw-value -> 'L'/'R' mapping if the hand values are not words, e.g. {0: 'R', 1: 'L'}
ALLOW_UNKNOWN_HAND = False

def load_json(p):
    with open(p, encoding='utf-8') as f:
        return json.load(f)

def iter_dicts(obj, path=()):
    if isinstance(obj, dict):
        yield path, obj
        for k, v in obj.items():
            yield from iter_dicts(v, path + (str(k),))
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            yield from iter_dicts(v, path + ('[%d]' % i,))

key_counter = collections.Counter()
for p in json_files[:3]:
    for _, d in iter_dicts(load_json(p)):
        key_counter.update(str(k) for k in d.keys())
print('most common keys in the first JSON files:', key_counter.most_common(25))

def pick_key(cands, prefer=(), avoid=()):
    cands = [k for k in cands if not any(a in k.lower() for a in avoid)]
    for pref in prefer:
        for k in cands:
            if pref in k.lower():
                return k
    return cands[0] if cands else None

keys = list(key_counter)
S_KEY = START_KEY or pick_key([k for k in keys if 'start' in k.lower()], prefer=('actual',))
E_KEY = END_KEY or pick_key([k for k in keys if 'end' in k.lower()], prefer=('end_frame', 'end'), avoid=('withdraw', 'extend'))
H_KEY = HAND_KEY or pick_key([k for k in keys if any(s in k.lower() for s in ('dominan', 'hand', 'orient'))], prefer=('dominan', 'hand'))
if H_KEY in ('LeftHand', 'RightHand'):   # these are tree levels (W1 -> U1 -> LeftHand), not fields
    H_KEY = None
print('using keys -> start:', S_KEY, '| end:', E_KEY, '| hand:', H_KEY)

def norm_hand(v):
    if v in HAND_MAP:
        return HAND_MAP[v]
    s = str(v).strip().lower()
    if s.startswith('l'):
        return 'L'
    if s.startswith('r'):
        return 'R'
    return None

def as_int(v):
    try:
        return int(float(v))
    except Exception:
        return None

def signature(path, d):
    parts = list(path) + [str(v) for v in d.values() if isinstance(v, (str, int))]
    return ' '.join(parts)

segs = []
for jp in json_files:
    for path, d in iter_dicts(load_json(jp)):
        if S_KEY not in d or E_KEY not in d:
            continue
        m = VID_RE.search(signature(path, d))
        view = ''
        if m:
            user, word, view = int(m[1]), int(m[2]), m[3].upper()
        else:
            uk = [k for k in d if 'user' in str(k).lower()]
            wk = [k for k in d if 'word' in str(k).lower()]
            user = as_int(re.sub(r'\D', '', str(d[uk[0]]))) if uk else None
            word = as_int(re.sub(r'\D', '', str(d[wk[0]]))) if wk else None
        if user is None or word is None or view not in ('', 'F'):
            continue
        starts, ends = d[S_KEY], d[E_KEY]
        path_hand = next((p for p in path if p in ('LeftHand', 'RightHand')), None)   # per-video hand from the JSON path
        hands = d.get(H_KEY) if H_KEY else path_hand
        if not isinstance(starts, list):
            starts, ends = [starts], [ends]
        if not isinstance(hands, list):
            hands = [hands] * len(starts)
        for i, (s, e) in enumerate(zip(starts, ends)):
            s, e = as_int(s), as_int(e)
            if s is None or e is None or e <= s:
                continue
            segs.append(dict(user=user, word=word, start=s, end=e,
                             hand_raw=hands[i] if i < len(hands) else None, src=jp.name))

segs = pd.DataFrame(segs)
print(len(segs), 'segments parsed')
assert len(segs) > 0, 'No segments parsed - send me the outputs of cells 5 and 6 so the adapter can be fixed.'
segs['hand'] = segs.hand_raw.map(norm_hand)
print('raw hand values:', segs.hand_raw.value_counts(dropna=False).head(8).to_dict())
n_unknown = int(segs.hand.isna().sum())
print('segments with unknown hand:', n_unknown)
if n_unknown and not ALLOW_UNKNOWN_HAND:
    raise RuntimeError('Hand dominance not found for some segments. Set HAND_KEY / HAND_MAP above, '
                       'or set ALLOW_UNKNOWN_HAND = True to treat them as right-handed (not recommended).')
segs['hand'] = segs.hand.fillna('R')

trials = segs.merge(videos[['user', 'word', 'stem', 'path']], on=['user', 'word'], how='inner')
trials = trials[trials.word.isin(WORD_IDS)].copy()
trials['cls'] = trials.word.map(WORD2CLS)
trials = trials.sort_values(['stem', 'start']).reset_index(drop=True)

per_video = trials.groupby('stem').size()
print('\ntrials matched to videos:', len(trials), '(paper: 9307)')
print('videos with at least one trial:', len(per_video), '/', len(videos))
print('segments per video (paper says 10-15):', per_video.describe()[['min', '25%', '50%', '75%', 'max']].to_dict())
print('left-handed trials:', int((trials.hand == 'L').sum()), '| right-handed:', int((trials.hand == 'R').sum()),
      '(paper: 1634 / 7673)')
cmp = pd.DataFrame({'ours': trials.groupby('user').size(), 'paper': pd.Series(TABLE6)}).fillna(0).astype(int)
cmp['diff'] = cmp.ours - cmp.paper
print('\ntrials per signer vs Table 6:')
print(cmp.T.to_string())

most common keys in the first JSON files: [('starting', 913), ('ending', 913), ('User', 75), ('Orientation', 75), ('Session', 75), ('View', 75), ('FrameRate', 75), ('FileName', 75), ('no_of_trials', 75), ('trials', 75), ('0', 75), ('1', 75), ('2', 75), ('3', 75), ('4', 75), ('5', 74), ('6', 74), ('7', 73), ('8', 73), ('9', 71), ('RightHand', 62), ('10', 50), ('11', 36), ('12', 26), ('13', 23)]
using keys -> start: starting | end: ending | hand: None
9297 segments parsed
raw hand values: {'RightHand': 7663, 'LeftHand': 1634}
segments with unknown hand: 0

trials matched to videos: 9297 (paper: 9307)
videos with at least one trial: 770 / 774
segments per video (paper says 10-15): {'min': 5.0, '25%': 10.0, '50%': 11.0, '75%': 13.0, 'max': 30.0}
left-handed trials: 1634 | right-handed: 7663 (paper: 1634 / 7673)

trials per signer vs Table 6:
        1    2    3    4    5    6    7    8    9    10   11   12   13   14   15  16   17   18
ours   975  744  583  637  600  815  107  639  656  383

In [7]:
# ---- Cell 6b: which annotated frames do the JSON 'starting' / 'ending' values use? ----
# Each trial in the txt files has four frames: "intention/actual_start  gesture_end/withdrawal", e.g. "39/44 86/88".
# The JSON keeps only one start and one end. This cell finds out which ones, by matching the two files.
LINE_RE = re.compile(r'^\s*(\d+)\s*/\s*(\d+)\s+(\d+)\s*/\s*(\d+)\s*$')
FILE_RE = re.compile(r'File:\s*(\S+)')
txt_trials = collections.defaultdict(list)          # stem -> [(intention, actual, gesture_end, withdrawal), ...]
for tp in txt_files:
    stem = None
    with open(tp, errors='ignore') as f:
        for line in f:
            fm = FILE_RE.search(line)
            if fm:
                stem = fm.group(1)
                continue
            lm = LINE_RE.match(line)
            if lm and stem:
                txt_trials[stem].append(tuple(int(g) for g in lm.groups()))
print('txt files parsed:', len(txt_trials), 'videos,', sum(len(v) for v in txt_trials.values()), 'trials')

counts = collections.Counter()
n_cmp = 0
for stem, g in trials.groupby('stem'):
    tt = sorted(txt_trials.get(stem, []))
    js = sorted(zip(g.start, g.end))
    if len(tt) != len(js):
        counts['trial count differs between txt and JSON'] += 1
        continue
    for (s, e), (a, b, c, d) in zip(js, tt):
        n_cmp += 1
        counts['start = intention' if s == a else 'start = actual start' if s == b else 'start = other'] += 1
        counts['end = gesture end' if e == c else 'end = withdrawal' if e == d else 'end = other'] += 1
print('trials compared:', n_cmp)
for k, v in sorted(counts.items()):
    print('   %-42s %d' % (k, v))
ex = trials.iloc[0]
print('\nexample:', ex.stem, '| JSON start/end:', ex.start, ex.end, '| txt first trial:', sorted(txt_trials.get(ex.stem, [None]))[0])
print('\nIf the JSON uses intention/withdrawal (the widest window), the clips include the lead-in and withdrawal motion.'
      '\nCheck which pair the authors\' code uses and keep the same pair, so your clips match theirs.')


txt files parsed: 774 videos, 9328 trials
trials compared: 9133
   end = gesture end                          201
   end = other                                8931
   end = withdrawal                           1
   start = actual start                       200
   start = intention                          1
   start = other                              8932
   trial count differs between txt and JSON   12

example: U10W10F | JSON start/end: 23 46 | txt first trial: (15, 21, 48, 52)

If the JSON uses intention/withdrawal (the widest window), the clips include the lead-in and withdrawal motion.
Check which pair the authors' code uses and keep the same pair, so your clips match theirs.


In [8]:
# ---- Cell 7: MediaPipe smoke test, left/right hand check, time estimate ----
import cv2
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
assert os.path.exists(MODEL_PATH), 'holistic_landmarker.task missing - see the message printed by cell 1.'
print('mediapipe', mp.__version__)
POSE_IDX = [0, 11, 12, 13, 14, 15, 16]   # nose, shoulders, elbows, wrists

def make_landmarker():
    opts = mp_vision.HolisticLandmarkerOptions(
        base_options=mp_python.BaseOptions(model_asset_path=MODEL_PATH),
        running_mode=mp_vision.RunningMode.VIDEO,
        min_pose_detection_confidence=0.5, min_pose_landmarks_confidence=0.5,
        min_hand_landmarks_confidence=0.5, min_face_detection_confidence=0.5,
        min_face_landmarks_confidence=0.5)
    return mp_vision.HolisticLandmarker.create_from_options(opts)

first = trials.groupby('stem').agg(path=('path', 'first'), f0=('start', 'min'), f1=('end', 'max')).reset_index()
sample = first.sample(min(3, len(first)), random_state=0)

lh_closer_to_pose_left, n_checked, n_frames, t_spent = [], 0, 0, 0.0
for r in sample.itertuples():
    h = make_landmarker()                      # new one per video: timestamps must increase within a landmarker
    cap = cv2.VideoCapture(r.path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    for _ in range(int(r.f0)):
        cap.grab()
    for k in range(60):
        ok, frame = cap.read()
        if not ok:
            break
        rgb = np.ascontiguousarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        t0 = time.time()
        res = h.detect_for_video(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb), k * 33)
        t_spent += time.time() - t0
        n_frames += 1
        if res.pose_landmarks and res.left_hand_landmarks:
            pl, pr = res.pose_landmarks[15], res.pose_landmarks[16]
            w = res.left_hand_landmarks[0]
            dl = (pl.x - w.x) ** 2 + (pl.y - w.y) ** 2
            dr = (pr.x - w.x) ** 2 + (pr.y - w.y) ** 2
            lh_closer_to_pose_left.append(dl < dr)
    cap.release()
    h.close()
print('video fps (last sample):', fps, '| frame size:', frame.shape if n_frames else None)
fps_proc = n_frames / max(t_spent, 1e-6)
print('MediaPipe speed: %.1f frames/s per process' % fps_proc)

if lh_closer_to_pose_left:
    frac = float(np.mean(lh_closer_to_pose_left))
    SWAP_HANDS = frac < 0.5
    print('left_hand wrist is nearest the pose LEFT wrist in %.0f%% of checked frames -> SWAP_HANDS = %s' % (100 * frac, SWAP_HANDS))
else:
    SWAP_HANDS = False
    print('No frame with both pose and a left hand in the sample; SWAP_HANDS left False. '
          'Only matters for naming - mirroring stays consistent either way.')

procs = max(1, os.cpu_count() or 1)
total_frames = int((first.f1 - first.f0 + 1).sum())
print('frames to process: %d | estimated time with %d processes: %.1f h'
      % (total_frames, procs, total_frames / (fps_proc * procs * 0.8) / 3600))

mediapipe 1.1.1


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1792948771.236010      91 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1792948771.269592      91 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1792948771.274226      93 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1792948771.275950      92 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1792948771.277345      94 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1792948771.318554     

video fps (last sample): 24.08294366956637 | frame size: (720, 1280, 3)
MediaPipe speed: 20.7 frames/s per process
left_hand wrist is nearest the pose LEFT wrist in 100% of checked frames -> SWAP_HANDS = False
frames to process: 673031 | estimated time with 4 processes: 2.8 h


In [9]:
# ---- Cell 8: extract keypoints for every video (resumable; one .npz per video) ----
EXTRACT_SCRIPT = r"""
import sys, json
import numpy as np
from pathlib import Path
from multiprocessing import get_context

POSE_IDX = [0, 11, 12, 13, 14, 15, 16]

def work(job):
    stem, path, f0, f1, swap, out_dir = job
    out = Path(out_dir) / (stem + '.npz')
    if out.exists():
        return stem, -1
    import os
    import cv2
    import mediapipe as mp
    from mediapipe.tasks import python as mp_python
    from mediapipe.tasks.python import vision as mp_vision
    cap = cv2.VideoCapture(path)
    for _ in range(f0):
        if not cap.grab():
            break
    coords, masks = [], []
    opts = mp_vision.HolisticLandmarkerOptions(
        base_options=mp_python.BaseOptions(model_asset_path=os.environ['HOLISTIC_MODEL']),
        running_mode=mp_vision.RunningMode.VIDEO,
        min_pose_detection_confidence=0.5, min_pose_landmarks_confidence=0.5,
        min_hand_landmarks_confidence=0.5, min_face_detection_confidence=0.5,
        min_face_landmarks_confidence=0.5)
    h = mp_vision.HolisticLandmarker.create_from_options(opts)
    for fi in range(f0, f1 + 1):
        ok, frame = cap.read()
        if not ok:
            break
        H, W = frame.shape[:2]
        if max(H, W) > 640:
            s = 640.0 / max(H, W)
            frame = cv2.resize(frame, (int(W * s), int(H * s)))
        rgb = np.ascontiguousarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        res = h.detect_for_video(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb), (fi - f0) * 33)
        c = np.zeros((49, 3), np.float32)
        m = np.zeros(49, np.uint8)
        if res.pose_landmarks:
            for k, idx in enumerate(POSE_IDX):
                lm = res.pose_landmarks[idx]
                c[k] = (lm.x, lm.y, lm.z)
                m[k] = 1
        lh, rh = res.left_hand_landmarks, res.right_hand_landmarks
        if swap:
            lh, rh = rh, lh
        for hand, sl, wrist in ((lh, slice(7, 28), 5), (rh, slice(28, 49), 6)):
            if hand:
                arr = np.array([[p.x, p.y, p.z] for p in hand], np.float32)
                if m[wrist]:
                    arr[:, 2] += c[wrist, 2]   # hand z is relative to the wrist; make it comparable to pose z
                c[sl] = arr
                m[sl] = 1
        coords.append(c)
        masks.append(m)
    h.close()
    cap.release()
    coords = np.stack(coords) if coords else np.zeros((0, 49, 3), np.float32)
    masks = np.stack(masks) if masks else np.zeros((0, 49), np.uint8)
    np.savez_compressed(out, coords=coords, mask=masks, f0=f0)
    return stem, len(coords)

if __name__ == '__main__':
    jobs = json.load(open(sys.argv[1]))
    procs = int(sys.argv[2])
    ctx = get_context('spawn')
    done = 0
    with ctx.Pool(procs, maxtasksperchild=20) as pool:
        for stem, n in pool.imap_unordered(work, jobs):
            done += 1
            print('%d/%d %s frames=%d' % (done, len(jobs), stem, n), flush=True)
    print('ALL DONE', flush=True)
"""
(CACHE / 'extract_kp.py').write_text(EXTRACT_SCRIPT)

jobs_df = first.copy()
jobs_df['f0'] = (jobs_df.f0 - 5).clip(lower=0)
jobs_df['f1'] = jobs_df.f1 + 5
if QUICK_TEST_VIDEOS:
    jobs_df = jobs_df.head(QUICK_TEST_VIDEOS)
jobs = [[r.stem, r.path, int(r.f0), int(r.f1), bool(SWAP_HANDS), str(KP_DIR)] for r in jobs_df.itertuples()]
json.dump(jobs, open(CACHE / 'jobs.json', 'w'))

if KP_SOURCE is not None:                  # reuse keypoints saved from an earlier run
    src = Path(KP_SOURCE)
    assert src.exists(), 'KP_SOURCE does not exist: %s' % src
    n_copied = 0
    for j in jobs:
        f, dst = src / (j[0] + '.npz'), KP_DIR / (j[0] + '.npz')
        if f.exists() and not dst.exists():
            shutil.copy(f, dst)
            n_copied += 1
    print('copied %d keypoint files from %s' % (n_copied, src))

todo = [j for j in jobs if not (KP_DIR / (j[0] + '.npz')).exists()]
print(len(jobs), 'videos in total,', len(todo), 'still to extract')

if todo:
    os.environ['HOLISTIC_MODEL'] = MODEL_PATH      # inherited by the extractor's worker processes
    log = open(CACHE / 'extract.log', 'w')
    proc = subprocess.Popen([sys.executable, str(CACHE / 'extract_kp.py'), str(CACHE / 'jobs.json'), str(procs)],
                            stdout=log, stderr=subprocess.STDOUT)
    t0 = time.time()
    while proc.poll() is None:          # blocks until finished, so "Save & Run All" works for long runs
        time.sleep(600)
        n_done = len(list(KP_DIR.glob('*.npz')))
        print('%.0f min | %d / %d videos done' % ((time.time() - t0) / 60, n_done, len(jobs)), flush=True)
    print('extractor exit code:', proc.returncode)
    print(''.join(open(CACHE / 'extract.log').readlines()[-5:]))
print('npz files:', len(list(KP_DIR.glob('*.npz'))))


copied 769 keypoint files from /kaggle/input/notebooks/rishta19/bdslw60-week-1/bdslw60_cache/keypoints
770 videos in total, 1 still to extract
10 min | 770 / 770 videos done
extractor exit code: 0
W0000 00:00:1792948794.849043     342 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1792948794.850061     341 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1792948794.900262     341 landmark_projection_calculator.cc:81] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.
770/770 U1W357F_2 frames=1135
ALL DONE

npz files: 770


In [10]:
# ---- Cell 9: preprocessing functions + self-tests on synthetic data ----
LEFT_BODY, RIGHT_BODY = [1, 3, 5], [2, 4, 6]
PERM = np.arange(N_NODES)
for l, r in zip(LEFT_BODY, RIGHT_BODY):
    PERM[l], PERM[r] = r, l
PERM[7:28], PERM[28:49] = np.arange(28, 49), np.arange(7, 28)

def fill_gaps(coords, mask, max_gap=MAX_GAP):
    # linear interpolation of gaps up to max_gap frames that are bounded by detections on both sides
    coords, mask = coords.copy(), mask.copy()
    for n in range(coords.shape[1]):
        idx = np.flatnonzero(mask[:, n])
        for a, b in zip(idx[:-1], idx[1:]):
            gap = b - a - 1
            if 1 <= gap <= max_gap:
                w = ((np.arange(a + 1, b) - a) / (b - a))[:, None]
                coords[a + 1:b, n] = (1 - w) * coords[a, n] + w * coords[b, n]
                mask[a + 1:b, n] = 1
    return coords, mask

def normalize(coords, mask):
    # centre on mid-shoulder, divide by shoulder width, per frame; missing nodes stay exactly 0
    ls, rs = coords[:, 1], coords[:, 2]
    ok = (mask[:, 1] > 0) & (mask[:, 2] > 0)
    if not ok.any():
        return None, None
    mid = (ls + rs) / 2
    width = np.linalg.norm((ls - rs)[:, :2], axis=1)
    good = ok & (width > 1e-4)
    if not good.any():
        return None, None
    mid_f = np.where(ok[:, None], mid, np.median(mid[ok], axis=0))
    width_f = np.where(good, width, np.median(width[good]))
    out = (coords - mid_f[:, None, :]) / width_f[:, None, None]
    out = out * mask[..., None]
    return out.astype(np.float32), mask

def mirror(coords, mask):
    c = coords[:, PERM].copy()
    c[..., 0] *= -1
    return c, mask[:, PERM]

def resample(coords, mask, t_out=T_OUT):
    # mask-weighted linear interpolation so a detected frame never blends with a zero-filled neighbour
    T = len(coords)
    if T == 1:
        return np.repeat(coords, t_out, 0), np.repeat(mask, t_out, 0)
    src = np.linspace(0, T - 1, t_out)
    lo = np.floor(src).astype(int)
    hi = np.minimum(lo + 1, T - 1)
    w = (src - lo)
    wv, wc = w[:, None], w[:, None, None]
    mf = mask.astype(np.float32)
    num = (1 - wc) * (coords[lo] * mf[lo][..., None]) + wc * (coords[hi] * mf[hi][..., None])
    den = (1 - wv) * mf[lo] + wv * mf[hi]
    mask_out = mask[np.rint(src).astype(int)]
    coords_out = np.where(mask_out[..., None] > 0, num / np.maximum(den[..., None], 1e-6), 0.0)
    return coords_out.astype(np.float32), mask_out

# ---- self-tests ----
rng = np.random.default_rng(0)
c = rng.normal(size=(30, N_NODES, 3)).astype(np.float32)
c[:, 1] = [0.6, 0.5, 0.0]; c[:, 2] = [0.4, 0.5, 0.0]          # shoulders: width 0.2
m = np.ones((30, N_NODES), np.uint8)
m[5:8, 10] = 0; m[12:20, 11] = 0; m[0:3, 12] = 0              # short gap, long gap, leading gap
c = c * m[..., None]
c2, m2 = fill_gaps(c, m)
assert m2[5:8, 10].all() and not m2[12:20, 11].any() and not m2[0:3, 12].any()
n, nm = normalize(c2, m2)
assert np.allclose(np.abs(n[:, 1, 0]), 0.5, atol=1e-4)      # shoulders end up at +-0.5
assert np.all(n[nm == 0] == 0)
mc, mm = mirror(n, nm)
mc2, mm2 = mirror(mc, mm)
assert np.allclose(mc2, n) and (mm2 == nm).all()
assert np.allclose(mc[:, 1, 0], -n[:, 2, 0])                 # left/right shoulders swapped and mirrored
const = np.ones((20, N_NODES, 3), np.float32); cm = np.ones((20, N_NODES), np.uint8)
cm[8:14, 20] = 0; const = const * cm[..., None]
rc, rm = resample(const, cm)
assert rc.shape == (T_OUT, N_NODES, 3) and rm.shape == (T_OUT, N_NODES)
assert np.allclose(rc[rm[:, 20] > 0, 20], 1.0)               # no partial values at mask borders
assert np.all(rc[rm == 0] == 0)
print('preprocessing self-tests passed')

preprocessing self-tests passed


In [11]:
# ---- Cell 10: build the cached arrays (two variants) and compare with the paper ----
# 'zero' = detection mask only, short gaps NOT filled (B0 baseline)   'fill' = short gaps interpolated (B1)
def build_variant(trials, fill):
    n = len(trials)
    X = np.zeros((n, 3, T_OUT, N_NODES), np.float32)
    M = np.zeros((n, T_OUT, N_NODES), np.uint8)
    ok = np.zeros(n, bool)
    hand_frames = hand_missing = 0
    cur_stem, kp = None, None
    for i, r in enumerate(trials.itertuples()):
        if r.stem != cur_stem:
            f = KP_DIR / (r.stem + '.npz')
            kp = np.load(f) if f.exists() else None
            if kp is not None:
                kp = dict(coords=kp['coords'], mask=kp['mask'], f0=int(kp['f0']))
            cur_stem = r.stem
        if kp is None:
            continue
        s, e = r.start - kp['f0'], r.end - kp['f0']
        if s < 0 or e >= len(kp['coords']) or e - s < 2:
            continue
        c, m = kp['coords'][s:e + 1], kp['mask'][s:e + 1]
        hand_frames += m[:, 7:].size
        hand_missing += int((m[:, 7:] == 0).sum())
        if fill:
            c, m = fill_gaps(c, m)
        c, m = normalize(c, m)
        if c is None:
            continue
        if r.hand == 'L':
            c, m = mirror(c, m)
        c, m = resample(c, m)
        X[i] = c.transpose(2, 0, 1)
        M[i] = m
        ok[i] = True
    return X, M, ok, hand_missing / max(hand_frames, 1)

t0 = time.time()
Xz, Mz, ok_z, miss_raw = build_variant(trials, fill=False)
Xf, Mf, ok_f, _ = build_variant(trials, fill=True)
keep = ok_z & ok_f
print('built in %.0f s | usable trials: %d / %d' % (time.time() - t0, keep.sum(), len(trials)))
print('hand points missing before filling: %.1f%% (paper: about 41%%)' % (100 * miss_raw))
hand_cols = slice(7, 49)
print('hand points missing after resampling: zero-fill %.1f%% | gap-filled %.1f%%'
      % (100 * (Mz[keep][:, :, hand_cols] == 0).mean(), 100 * (Mf[keep][:, :, hand_cols] == 0).mean()))

meta = trials[keep].reset_index(drop=True)
Xz, Mz, Xf, Mf = Xz[keep], Mz[keep], Xf[keep], Mf[keep]
y = meta.cls.values.astype(np.int64)
np.save(CACHE / 'X_zero.npy', Xz); np.save(CACHE / 'M_zero.npy', Mz)
np.save(CACHE / 'X_fill.npy', Xf); np.save(CACHE / 'M_fill.npy', Mf)
np.save(CACHE / 'y.npy', y)
meta.to_csv(CACHE / 'meta.csv', index=False)

cmp = pd.DataFrame({'ours': meta.groupby('user').size(), 'paper': pd.Series(TABLE6)}).fillna(0).astype(int)
cmp['diff'] = cmp.ours - cmp.paper
print('\ntrials per signer vs Table 6 (diff should be 0 or very small):')
print(cmp.T.to_string())
print('test signers U4+U8 trials:', int(meta.user.isin([4, 8]).sum()), '(paper: 1276)')
print('classes present:', len(set(y)), '/ 60')

built in 22 s | usable trials: 9296 / 9297
hand points missing before filling: 42.3% (paper: about 41%)
hand points missing after resampling: zero-fill 43.1% | gap-filled 39.4%

trials per signer vs Table 6 (diff should be 0 or very small):
        1    2    3    4    5    6    7    8    9    10   11   12   13   14   15  16   17   18
ours   974  744  583  637  600  815  107  639  656  383  771  655  621  112  655  97  116  131
paper  975  744  583  637  600  815  107  639  656  383  771  655  621  112  655  97  116  141
diff    -1    0    0    0    0    0    0    0    0    0    0    0    0    0    0   0    0  -10
test signers U4+U8 trials: 1276 (paper: 1276)
classes present: 60 / 60


In [12]:
# ---- Cell 10b: duplicate check - is any recording stored under two different signers? ----
# Identical keypoints under two user ids would leak a "test" signer into training and inflate scores.
def arr_hash(a, decimals=4):
    return hashlib.md5(np.round(np.asarray(a, np.float32), decimals).tobytes()).hexdigest()

# 1) whole videos: identical raw keypoint arrays
vid_hash = collections.defaultdict(list)
for f in sorted(KP_DIR.glob('*.npz')):
    z = np.load(f)
    if len(z['coords']):
        vid_hash[arr_hash(z['coords'])].append(f.stem)
vid_dups = [v for v in vid_hash.values() if len(v) > 1]
print('videos with identical keypoints:', vid_dups if vid_dups else 'none')

# 2) single trials: identical cached arrays, and whether the copies belong to different signers
trial_hash = collections.defaultdict(list)
for i in range(len(Xf)):
    trial_hash[arr_hash(Xf[i])].append(i)
same, cross = 0, []
for ids in trial_hash.values():
    if len(ids) > 1:
        us = sorted(set(meta.user.values[ids]))
        if len(us) > 1:
            cross.append((meta.stem.values[ids].tolist(), us))
        else:
            same += 1
print('groups of identical trials within one signer:', same)
print('groups of identical trials across different signers:', len(cross))
for stems, us in cross[:10]:
    print('   ', stems, 'signers', us)
if cross:
    print('ACTION: these copies sit under different signers. Remove one copy before trusting any validation score.')


videos with identical keypoints: none
groups of identical trials within one signer: 7
groups of identical trials across different signers: 0


In [13]:
# ---- Cell 11: split files (P1 official split, inner model-selection folds, 18 leave-one-signer-out folds) ----
users = sorted(int(u) for u in meta.user.unique())
clips_per_user = meta.groupby('user').size().to_dict()
lh_per_user = meta[meta.hand == 'L'].groupby('user').size().reindex(users, fill_value=0).to_dict()
def idx_of(us):
    return np.flatnonzero(meta.user.isin(us).values)

# P1: final runs. Train 15 signers, pick the checkpoint on U5, test on U4+U8 once (week 3 only).
p1 = {'train': idx_of([u for u in users if u not in TEST_SIGNERS + (P1_VAL_SIGNER,)]),
      'val': idx_of([P1_VAL_SIGNER]), 'test': idx_of(list(TEST_SIGNERS))}
print('P1 sizes -> train %d | val (U%d) %d | test (U4+U8) %d   (paper/plan: ~7431 / 600 / 1276)'
      % (len(p1['train']), P1_VAL_SIGNER, len(p1['val']), len(p1['test'])))

# Inner folds: all model decisions (B0 vs B1 vs B2 ...). Each holds out one of INNER_VAL_SIGNERS and trains
# on every other non-test signer. U4/U8 appear in none of them.
inner = {}
for v in INNER_VAL_SIGNERS:
    tr_users = [u for u in users if u not in TEST_SIGNERS + (v,)]
    inner[v] = {'train': idx_of(tr_users), 'val': idx_of([v])}
    assert not set(meta.user.values[inner[v]['train']]) & set(TEST_SIGNERS + (v,))
    print('inner fold val U%-2d: train %d | val %d (left-handed %d)'
          % (v, len(inner[v]['train']), len(inner[v]['val']), lh_per_user[v]))

# LOSO: a TEST-TIME protocol (two of its folds test on U4 and U8). Run it in week 3 only.
# Each fold's validation signer comes from the signers with >= MIN_VAL_CLIPS clips.
big = [u for u in users if clips_per_user[u] >= MIN_VAL_CLIPS]
loso = {}
for k, s in enumerate(users):
    cands = [u for u in big if u != s]
    v = cands[k % len(cands)]
    loso[s] = {'train': idx_of([u for u in users if u not in (s, v)]), 'val': idx_of([v]), 'test': idx_of([s])}
print('LOSO validation signers (test -> val):', {s: int(meta.user.values[d['val'][0]]) for s, d in loso.items()})
assert all(len(d['val']) >= MIN_VAL_CLIPS for d in loso.values())

splits = {'P1': {k: v.tolist() for k, v in p1.items()},
          'INNER': {str(v): {k: a.tolist() for k, a in d.items()} for v, d in inner.items()},
          'LOSO': {str(s): {k: v.tolist() for k, v in d.items()} for s, d in loso.items()}}
json.dump(splits, open(CACHE / 'splits.json', 'w'))
print('LOSO folds written:', len(loso))
print('files in cache:', sorted(p.name for p in CACHE.iterdir() if p.is_file()))


P1 sizes -> train 7420 | val (U5) 600 | test (U4+U8) 1276   (paper/plan: ~7431 / 600 / 1276)
inner fold val U1 : train 7046 | val 974 (left-handed 653)
inner fold val U10: train 7637 | val 383 (left-handed 148)
inner fold val U13: train 7399 | val 621 (left-handed 186)
LOSO validation signers (test -> val): {1: 2, 2: 3, 3: 4, 4: 5, 5: 6, 6: 8, 7: 8, 8: 11, 9: 12, 10: 12, 11: 15, 12: 1, 13: 2, 14: 2, 15: 4, 16: 4, 17: 5, 18: 6}
LOSO folds written: 18
files in cache: ['M_fill.npy', 'M_zero.npy', 'X_fill.npy', 'X_zero.npy', 'extract.log', 'extract_kp.py', 'jobs.json', 'meta.csv', 'splits.json', 'y.npy']


In [14]:
# ---- Cell 12: 49-node graph, augmentation, bone/motion streams (numpy only) ----
HAND_EDGES = [(0, 1), (1, 2), (2, 3), (3, 4), (0, 5), (5, 6), (6, 7), (7, 8), (5, 9), (9, 10), (10, 11), (11, 12),
              (9, 13), (13, 14), (14, 15), (15, 16), (13, 17), (17, 18), (18, 19), (19, 20), (0, 17)]
BODY_EDGES = [(0, 1), (0, 2), (1, 2), (1, 3), (2, 4), (3, 5), (4, 6)]
EDGES = (BODY_EDGES + [(a + 7, b + 7) for a, b in HAND_EDGES] + [(a + 28, b + 28) for a, b in HAND_EDGES]
         + [(5, 7), (6, 28)])      # hand wrists attached to the body wrists

def build_A(edges, V=N_NODES):
    A = np.eye(V, dtype=np.float32)
    for a, b in edges:
        A[a, b] = A[b, a] = 1
    d = A.sum(1) ** -0.5
    return (d[:, None] * A * d[None, :]).astype(np.float32)

def build_parents(edges, V=N_NODES, root=0):
    adj = collections.defaultdict(list)
    for a, b in edges:
        adj[a].append(b); adj[b].append(a)
    parent = [-1] * V
    parent[root] = root
    queue = [root]
    while queue:
        u = queue.pop(0)
        for w in adj[u]:
            if parent[w] == -1:
                parent[w] = u
                queue.append(w)
    assert all(p >= 0 for p in parent), 'graph is not connected'
    return np.array(parent)

A_NORM = build_A(EDGES)
PARENT = build_parents(EDGES)

def time_warp(x, m, gamma):
    # Monotonic warp that keeps the whole sign: frame t samples the clip at (T-1) * (t/(T-1))**gamma.
    # gamma < 1: the start is played slower and the end faster; gamma > 1: the opposite. First and last frames stay.
    C, T, V = x.shape
    src = (T - 1) * np.linspace(0, 1, T) ** gamma
    lo = np.floor(src).astype(int)
    hi = np.minimum(lo + 1, T - 1)
    w = src - lo
    mf = m.astype(np.float32)
    num = (1 - w)[None, :, None] * x[:, lo] * mf[lo][None] + w[None, :, None] * x[:, hi] * mf[hi][None]
    den = (1 - w)[:, None] * mf[lo] + w[:, None] * mf[hi]
    m_out = m[np.rint(src).astype(int)]
    x_out = np.where(m_out[None] > 0, num / np.maximum(den[None], 1e-6), 0.0)
    return x_out.astype(np.float32), m_out

def augment(x, m, rng, p_rot=0.5, p_scale=0.5, p_speed=0.5, p_hdrop=0.2):
    # x: (C, T, V) coordinates, m: (T, V) detection mask. Training only.
    C, T, V = x.shape
    x, m = x.copy(), m.copy()
    if rng.random() < p_speed:                       # rhythm change; keeps the whole sign (see time_warp)
        x, m = time_warp(x, m, float(np.exp(rng.uniform(np.log(0.8), np.log(1.25)))))
    if rng.random() < p_rot:                         # rotation +-15 degrees in the image plane
        th = np.deg2rad(rng.uniform(-15, 15))
        c, s_ = np.cos(th), np.sin(th)
        x0, x1 = x[0].copy(), x[1].copy()
        x[0], x[1] = c * x0 - s_ * x1, s_ * x0 + c * x1
    if rng.random() < p_scale:                       # scale 0.8-1.2
        x = x * rng.uniform(0.8, 1.2)
    if rng.random() < p_hdrop:                       # one hand disappears for a short span
        sl = slice(7, 28) if rng.random() < 0.5 else slice(28, 49)
        L = int(rng.integers(8, 25))
        st = int(rng.integers(0, max(1, T - L)))
        x[:, st:st + L, sl] = 0
        m[st:st + L, sl] = 0
    return (x * m[None]).astype(np.float32), m

def make_stream(x, m, stream, use_mask, use_z=True):
    # returns features (C', T, V) and the matching mask (T, V)
    if stream == 'joint':
        f, fm = x, m
    elif stream == 'bone':
        f = x - x[:, :, PARENT]
        fm = m * m[:, PARENT]
    elif stream == 'joint_motion':
        f = np.zeros_like(x); fm = np.zeros_like(m)
        f[:, :-1] = x[:, 1:] - x[:, :-1]
        fm[:-1] = m[1:] * m[:-1]
    elif stream == 'bone_motion':
        b, bm = make_stream(x, m, 'bone', False)
        f = np.zeros_like(b); fm = np.zeros_like(bm)
        f[:, :-1] = b[:, 1:] - b[:, :-1]
        fm[:-1] = bm[1:] * bm[:-1]
    else:
        raise ValueError(stream)
    if use_mask:
        f = f * fm[None]
    if not use_z:
        f = f[:2]
    return f.astype(np.float32), fm

# ---- self-tests ----
assert A_NORM.shape == (N_NODES, N_NODES) and np.allclose(A_NORM, A_NORM.T)
assert PARENT[0] == 0 and len(set(PARENT.tolist())) > 10
_rng = np.random.default_rng(1)
_x = _rng.normal(size=(3, T_OUT, N_NODES)).astype(np.float32)
_m = (_rng.random((T_OUT, N_NODES)) > 0.3).astype(np.uint8)
_x = _x * _m[None]
for _s in ('joint', 'bone', 'joint_motion', 'bone_motion'):
    for _um in (False, True):
        _f, _fm = make_stream(_x, _m, _s, _um)
        assert _f.shape == (3, T_OUT, N_NODES) and _fm.shape == (T_OUT, N_NODES)
        if _um:
            assert np.all(_f[:, _fm == 0] == 0)
_w, _wm = time_warp(_x, _m, 0.8)
assert _w.shape == _x.shape and (_wm[0] == _m[0]).all() and (_wm[-1] == _m[-1]).all()   # whole sign kept
assert np.allclose(time_warp(_x, _m, 1.0)[0], _x)                                       # gamma 1 = identity
_a, _am = augment(_x, _m, np.random.default_rng(2), 1, 1, 1, 1)
assert _a.shape == _x.shape and np.all(_a[:, _am == 0] == 0)
print('graph:', len(EDGES), 'edges |', 'self-tests passed')

graph: 51 edges | self-tests passed


In [15]:
# ---- Cell 13: dataset + compact ST-GCN-style model + training loop (PyTorch) ----
import torch, torch.nn as nn, torch.nn.functional as F
from sklearn.metrics import f1_score

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

class BdslDataset(torch.utils.data.Dataset):
    def __init__(self, X, M, y, idx, stream='joint', use_mask=False, aug=False, use_z=True):
        self.X, self.M, self.y, self.idx = X, M, y, idx
        self.stream, self.use_mask, self.aug, self.use_z = stream, use_mask, aug, use_z
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, k):
        i = self.idx[k]
        x, m = np.array(self.X[i]), np.array(self.M[i])
        if self.aug:
            rng = np.random.default_rng(int(torch.randint(0, 2 ** 31 - 1, (1,))))
            x, m = augment(x, m, rng)
        f, fm = make_stream(x, m, self.stream, self.use_mask, self.use_z)
        return torch.from_numpy(f), torch.from_numpy(fm.astype(np.float32)), int(self.y[i])

class STBlock(nn.Module):
    def __init__(self, cin, cout, A, stride=1, dropout=0.2):
        super().__init__()
        self.register_buffer('A', torch.tensor(A, dtype=torch.float32))
        self.B = nn.Parameter(torch.zeros(A.shape, dtype=torch.float32))   # learnable adjacency offset
        self.gcn = nn.Conv2d(cin, cout, 1)
        self.bn1 = nn.BatchNorm2d(cout)
        self.tcn = nn.Sequential(nn.Conv2d(cout, cout, (9, 1), (stride, 1), (4, 0)),
                                 nn.BatchNorm2d(cout), nn.Dropout(dropout))
        if cin == cout and stride == 1:
            self.res = nn.Identity()
        else:
            self.res = nn.Sequential(nn.Conv2d(cin, cout, 1, (stride, 1)), nn.BatchNorm2d(cout))
    def forward(self, x):                                   # x: (N, C, T, V)
        y = self.gcn(x)
        y = torch.einsum('nctv,vw->nctw', y, self.A + self.B)
        y = F.relu(self.bn1(y))
        y = self.tcn(y)
        return F.relu(y + self.res(x))

class GCNNet(nn.Module):
    def __init__(self, in_ch, n_cls, A, V=N_NODES, widths=(64, 64, 128, 128, 256, 256), strides=(1, 1, 2, 1, 2, 1),
                 masked_pool=False):
        super().__init__()
        self.masked_pool = masked_pool
        self.data_bn = nn.BatchNorm1d(in_ch * V)
        blocks, c = [], in_ch
        for w, s in zip(widths, strides):
            blocks.append(STBlock(c, w, A, s))
            c = w
        self.blocks = nn.ModuleList(blocks)
        self.drop = nn.Dropout(0.3)
        self.fc = nn.Linear(c, n_cls)
    def forward(self, x, mask=None):                        # mask: (N, T, V), 1 = detected
        N, C, T, V = x.shape
        x = self.data_bn(x.permute(0, 1, 3, 2).reshape(N, C * V, T))
        x = x.reshape(N, C, V, T).permute(0, 1, 3, 2).contiguous()
        for b in self.blocks:
            x = b(x)
        if self.masked_pool and mask is not None:          # average only over detected nodes and frames
            w = F.adaptive_max_pool2d(mask[:, None].float(), x.shape[2:])   # (N, 1, T', V)
            pooled = (x * w).sum(dim=(2, 3)) / w.sum(dim=(2, 3)).clamp(min=1.0)
        else:
            pooled = x.mean(dim=(2, 3))
        return self.fc(self.drop(pooled))

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    logits, ys = [], []
    for x, m, y_ in loader:
        logits.append(model(x.to(device), m.to(device)).float().cpu()); ys.append(y_)
    logits, ys = torch.cat(logits), torch.cat(ys)
    top1 = (logits.argmax(1) == ys).float().mean().item()
    top5 = (logits.topk(5, dim=1).indices == ys[:, None]).any(1).float().mean().item()
    f1 = f1_score(ys.numpy(), logits.argmax(1).numpy(), average='macro')
    return top1, top5, f1

def run_experiment(name, variant, use_mask=False, stream='joint', epochs=40, seed=0, lr=0.1, bs=64,
                   aug=True, use_z=True, split=None, tag='', masked_pool=False):
    split = split or p1
    set_seed(seed)
    X = np.load(CACHE / ('X_%s.npy' % variant), mmap_mode='r')
    M = np.load(CACHE / ('M_%s.npy' % variant), mmap_mode='r')
    tr = BdslDataset(X, M, y, split['train'], stream, use_mask, aug, use_z)
    va = BdslDataset(X, M, y, split['val'], stream, use_mask, False, use_z)
    tl = torch.utils.data.DataLoader(tr, batch_size=bs, shuffle=True, num_workers=2, drop_last=True, pin_memory=True)
    vl = torch.utils.data.DataLoader(va, batch_size=128, shuffle=False, num_workers=2)
    model = GCNNet(3 if use_z else 2, 60, A_NORM, masked_pool=masked_pool).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, nesterov=True, weight_decay=1e-4)
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)
    amp = device == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=amp)
    best = dict(f1=-1, top1=0, top5=0, epoch=0)
    print('%s %s seed %d | stream=%s variant=%s mask=%s masked_pool=%s | %.2fM params | train %d val %d'
          % (name, tag, seed, stream, variant, use_mask, masked_pool, n_params / 1e6, len(tr), len(va)))
    for ep in range(epochs):
        cur = lr * (ep + 1) / 5 if ep < 5 else 0.5 * lr * (1 + math.cos(math.pi * (ep - 5) / max(1, epochs - 5)))
        for g in opt.param_groups:
            g['lr'] = cur
        model.train()
        t0, tot, n = time.time(), 0.0, 0
        for x, m, y_ in tl:
            x, m, y_ = x.to(device), m.to(device), y_.to(device)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=amp):
                loss = crit(model(x, m), y_)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            tot += loss.item() * len(y_); n += len(y_)
        top1, top5, f1 = evaluate(model, vl)
        if f1 > best['f1']:
            best = dict(f1=f1, top1=top1, top5=top5, epoch=ep + 1)
            torch.save(model.state_dict(), CACHE / ('%s%s_seed%d_best.pt' % (name, '_' + tag if tag else '', seed)))
        if ep % 5 == 0 or ep == epochs - 1:
            print('  ep %3d | loss %.3f | val top1 %.3f top5 %.3f macroF1 %.3f | %.0fs'
                  % (ep + 1, tot / n, top1, top5, f1, time.time() - t0))
    return dict(name=name, seed=seed, params_M=round(n_params / 1e6, 2), best_epoch=best['epoch'],
                val_top1=round(best['top1'], 4), val_top5=round(best['top5'], 4), val_macroF1=round(best['f1'], 4))

# quick shape check on a few real samples before the long runs
_X = np.load(CACHE / 'X_fill.npy', mmap_mode='r'); _M = np.load(CACHE / 'M_fill.npy', mmap_mode='r')
_ds = BdslDataset(_X, _M, y, p1['train'][:8], 'joint', False, True)
_x, _m, _y = next(iter(torch.utils.data.DataLoader(_ds, batch_size=8)))
_net = GCNNet(3, 60, A_NORM)
print('input', tuple(_x.shape), '-> logits', tuple(_net(_x, _m).shape))
_netm = GCNNet(3, 60, A_NORM, masked_pool=True)
print('masked pooling ok ->', tuple(_netm(_x, _m).shape))

device: cuda
input (8, 3, 64, 49) -> logits (8, 60)
masked pooling ok -> (8, 60)


In [16]:
# ---- Cell 14: B0 (zero-filled) vs B1 (+ short-gap filling), joint stream, inner folds x seeds ----
# Decisions use the mean over INNER_VAL_SIGNERS x SEEDS. One run takes about 9 min on a T4 at 40 epochs,
# so the full grid below (2 configs x 3 folds x 3 seeds = 18 runs) takes about 2.7 h.
# For a first look, set SEEDS = (0,) (about 55 min).
EPOCHS = 40
SEEDS = (0, 1, 2)
CONFIGS = [dict(name='B0_zerofill_joint', variant='zero'),
           dict(name='B1_gapfill_joint', variant='fill')]

res_path = CACHE / 'week1_inner_results.csv'
done = pd.read_csv(res_path) if res_path.exists() else pd.DataFrame()
results = done.to_dict('records')
for cfg in CONFIGS:
    for v in INNER_VAL_SIGNERS:
        for seed in SEEDS:
            if len(done) and ((done.name == cfg['name']) & (done.val_signer == v) & (done.seed == seed)).any():
                continue                                   # resumable: skip runs already in the csv
            r = run_experiment(cfg['name'], cfg['variant'], epochs=EPOCHS, seed=seed, split=inner[v], tag='U%d' % v)
            r['val_signer'] = v
            results.append(r)
            pd.DataFrame(results).to_csv(res_path, index=False)

res_df = pd.DataFrame(results)
summary = (res_df.groupby('name')[['val_top1', 'val_macroF1']]
           .agg(['mean', 'std', 'count']).round(4))
print('mean over inner folds x seeds:')
print(summary.to_string())

# paired difference per (fold, seed): the honest way to see whether B1 beats B0
piv = res_df.pivot_table(index=['val_signer', 'seed'], columns='name', values='val_macroF1')
if {'B0_zerofill_joint', 'B1_gapfill_joint'} <= set(piv.columns):
    d = (piv['B1_gapfill_joint'] - piv['B0_zerofill_joint']).dropna()
    print('\nB1 - B0 macro-F1 per (fold, seed):', d.round(4).tolist())
    print('mean %.4f | std %.4f | B1 better in %d of %d pairs' % (d.mean(), d.std(), int((d > 0).sum()), len(d)))
    print('per validation signer:', res_df.groupby(['val_signer', 'name']).val_macroF1.mean().unstack().round(4).to_dict('index'))
res_df


B0_zerofill_joint U1 seed 0 | stream=joint variant=zero mask=False masked_pool=False | 1.75M params | train 7046 val 974
  ep   1 | loss 2.877 | val top1 0.407 top5 0.708 macroF1 0.329 | 14s
  ep   6 | loss 1.117 | val top1 0.783 top5 0.953 macroF1 0.757 | 13s
  ep  11 | loss 0.918 | val top1 0.820 top5 0.956 macroF1 0.799 | 14s
  ep  16 | loss 0.851 | val top1 0.812 top5 0.954 macroF1 0.793 | 14s
  ep  21 | loss 0.815 | val top1 0.816 top5 0.951 macroF1 0.799 | 14s
  ep  26 | loss 0.795 | val top1 0.829 top5 0.957 macroF1 0.806 | 14s
  ep  31 | loss 0.781 | val top1 0.834 top5 0.944 macroF1 0.810 | 14s
  ep  36 | loss 0.776 | val top1 0.840 top5 0.943 macroF1 0.818 | 14s
  ep  40 | loss 0.776 | val top1 0.833 top5 0.950 macroF1 0.816 | 14s
B0_zerofill_joint U1 seed 1 | stream=joint variant=zero mask=False masked_pool=False | 1.75M params | train 7046 val 974
  ep   1 | loss 2.836 | val top1 0.368 top5 0.762 macroF1 0.315 | 14s
  ep   6 | loss 1.129 | val top1 0.772 top5 0.932 macroF1 

,name,seed,params_M,best_epoch,val_top1,val_top5,val_macroF1,val_signer
0,B0_zerofill_joint,0,1.75,25,0.8542,0.9435,0.8327,1
1,B0_zerofill_joint,1,1.75,38,0.8203,0.9661,0.8068,1
2,B0_zerofill_joint,2,1.75,24,0.8275,0.9733,0.8189,1
3,B0_zerofill_joint,0,1.75,26,0.9426,0.9791,0.8678,10
4,B0_zerofill_joint,1,1.75,38,0.9556,0.9896,0.9009,10
5,B0_zerofill_joint,2,1.75,28,0.9478,0.9791,0.9239,10
6,B0_zerofill_joint,0,1.75,18,0.8744,0.9469,0.8483,13
7,B0_zerofill_joint,1,1.75,28,0.8680,0.9630,0.8459,13
8,B0_zerofill_joint,2,1.75,39,0.8808,0.9565,0.8582,13
9,B1_gapfill_joint,0,1.75,25,0.8131,0.9487,0.7920,1
